# Assignment 2: AI-generated vs Real photos with CLIP (SOTA) + explainability
**Method:** frozen **CLIP ViT-L/14** image embeddings + linear probe (UnivFD-style, Ojha et al., CVPR 2023).
**Baseline (A1):** ResNet50 (ImageNet) features + the same classifier head, and your A1 Keras numbers.
**Sections:** data, features, main comparison, ablations, learning curves, robustness, explainability, latency, cross-generator test (SDXL-Turbo), export for the app.

Runtime: *Runtime -> Change runtime type -> T4 GPU*. Dataset layout is the same as A1:
`MyDrive/ai_vs_real/<ai-class-folder>/` and `MyDrive/ai_vs_real/<real-class-folder>/`.

## 0. Setup

In [ ]:
!pip -q install open_clip_torch

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/ai_vs_real'      # same folder as A1
OUT = '/content/drive/MyDrive/ai_vs_real_A2_results'  # figures and tables for the report
SEED = 42
STANDARDIZE = False   # set True to re-save every image as JPEG (see section 1)
import os; os.makedirs(OUT, exist_ok=True)

In [ ]:
%%writefile detector.py
# Shared helpers for the A2 project: image encoders, occlusion heatmap, nearest neighbours.
# Used by both the Colab notebook and the Gradio app (app.py).
import numpy as np
import torch
from PIL import Image, ImageOps, ImageDraw


def load_image(path):
    im = Image.open(path)
    im.load()
    return im.convert("RGB")


class _Encoder:
    """Base class: turns a list of PIL images into a feature matrix."""
    l2 = False

    def _init_device(self, device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.dtype = torch.float16 if self.device == "cuda" else torch.float32

    def move(self, device):
        self.device = device
        self.dtype = torch.float16 if device == "cuda" else torch.float32
        self.model.to(device=device, dtype=self.dtype)
        return self

    @torch.no_grad()
    def encode(self, imgs, bs=64):
        out = []
        for i in range(0, len(imgs), bs):
            x = torch.stack([self.preprocess(im) for im in imgs[i:i + bs]])
            x = x.to(self.device, self.dtype)
            f = self._forward(x).float()
            if self.l2:
                f = f / f.norm(dim=-1, keepdim=True)
            out.append(f.cpu().numpy())
        return np.concatenate(out)


class ClipEncoder(_Encoder):
    """Frozen CLIP image encoder (open_clip). L2-normalised embeddings."""
    l2 = True

    def __init__(self, arch="ViT-L-14", pretrained="openai", device=None):
        import open_clip
        self._init_device(device)
        model, _, preprocess = open_clip.create_model_and_transforms(arch, pretrained=pretrained)
        self.model = model.eval().to(device=self.device, dtype=self.dtype)
        self.preprocess = preprocess
        self.name = f"CLIP {arch} ({pretrained})"

    def _forward(self, x):
        return self.model.encode_image(x)


class ResNetEncoder(_Encoder):
    """ImageNet ResNet50 feature extractor (2048-d). Used as the A1-style baseline."""

    def __init__(self, device=None):
        import torchvision
        self._init_device(device)
        w = torchvision.models.ResNet50_Weights.IMAGENET1K_V2
        m = torchvision.models.resnet50(weights=w)
        m.fc = torch.nn.Identity()
        self.model = m.eval().to(device=self.device, dtype=self.dtype)
        self.preprocess = w.transforms()
        self.name = "ResNet50 (ImageNet)"

    def _forward(self, x):
        return self.model(x)


def occlusion_heatmap(encoder, clf, img, grid=7, target=None, fill=(124, 116, 104)):
    """Model-agnostic explanation: slide a grey square over the image and measure
    how much the probability of the target class drops. Big drop = important region.
    Returns (heat[grid, grid], base_image_224)."""
    base = ImageOps.fit(img.convert("RGB"), (224, 224), Image.BICUBIC)
    p0 = clf.predict_proba(encoder.encode([base]))[0]
    t = int(p0.argmax()) if target is None else int(target)
    step = 224 / grid
    variants = []
    for r in range(grid):
        for c in range(grid):
            v = base.copy()
            ImageDraw.Draw(v).rectangle(
                [int(c * step), int(r * step), int((c + 1) * step) - 1, int((r + 1) * step) - 1], fill=fill)
            variants.append(v)
    ps = clf.predict_proba(encoder.encode(variants))[:, t]
    heat = (p0[t] - ps).reshape(grid, grid)
    return heat, base


def heat_overlay(base, heat, strength=0.65):
    """Blend a coarse heatmap (positive = supports the decision) onto the image."""
    import matplotlib
    h = np.clip(heat, 0, None)
    h = h / (h.max() + 1e-8)
    h_img = Image.fromarray((h * 255).astype(np.uint8)).resize(base.size, Image.BICUBIC)
    a = np.asarray(h_img, dtype=np.float32)[..., None] / 255.0
    rgb = matplotlib.colormaps["jet"](a[..., 0])[..., :3] * 255.0
    out = np.asarray(base.convert("RGB"), dtype=np.float32) * (1 - a * strength) + rgb * (a * strength)
    return Image.fromarray(out.clip(0, 255).astype(np.uint8))


def nearest_neighbors(q, gallery, k=5):
    """Cosine-similarity neighbours (embeddings are L2-normalised)."""
    sims = gallery @ q
    idx = np.argsort(-sims)[:k]
    return idx, sims[idx]

In [ ]:
import os, io, time, random, shutil, joblib
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
import torchvision.transforms as T
from PIL import Image, ImageFilter, ImageOps
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, roc_auc_score,
                             confusion_matrix, ConfusionMatrixDisplay, log_loss)
from detector import load_image, ClipEncoder, ResNetEncoder, occlusion_heatmap, heat_overlay, nearest_neighbors

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('torch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (CPU)')

def savefig(name):
    plt.savefig(f'{OUT}/{name}.png', dpi=200, bbox_inches='tight')

## 1. Dataset (A1 data, updated) and sanity checks
The dataset is 50% synthetic by construction (AI class) and 50% real. Before trusting any accuracy, check that the two classes
do not differ in trivial ways (file format, resolution, file size). Otherwise a model can learn the *format* instead of the *content*.

In [ ]:
class_names = sorted(d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d)))
assert len(class_names) == 2, f'Expected 2 class folders, found {class_names}'
AI_IDX = [i for i, c in enumerate(class_names) if 'ai' in c.lower()][0]
print('Classes:', class_names, '| AI class index:', AI_IDX)

def list_images(root):
    paths, labels = [], []
    for idx, c in enumerate(class_names):
        for f in sorted(os.listdir(os.path.join(root, c))):
            if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
                paths.append(os.path.join(root, c, f)); labels.append(idx)
    return np.array(paths), np.array(labels)

if STANDARDIZE:
    STD_DIR = '/content/ai_vs_real_std'
    for c in class_names: os.makedirs(f'{STD_DIR}/{c}', exist_ok=True)
    p0, l0 = list_images(DATA_DIR)
    for i, (p, l) in enumerate(zip(p0, l0)):
        im = load_image(p); im.thumbnail((768, 768))
        im.save(f'{STD_DIR}/{class_names[l]}/{i:04d}.jpg', quality=95)
    DATA_DIR = STD_DIR

paths, labels = list_images(DATA_DIR)
print({c: int((labels == i).sum()) for i, c in enumerate(class_names)})

In [ ]:
rows = []
for p, y in zip(paths, labels):
    im = Image.open(p)
    rows.append(dict(cls=class_names[y], w=im.width, h=im.height, ext=os.path.splitext(p)[1].lower(), kb=os.path.getsize(p) / 1024))
df = pd.DataFrame(rows)
df['aspect'] = df.w / df.h
print(df.groupby('cls')[['w', 'h', 'aspect', 'kb']].agg(['mean', 'std']).round(2))
print(pd.crosstab(df.cls, df.ext))

If the table shows a big difference (e.g. all AI images are square PNG, all real are JPEG), set `STANDARDIZE = True` at the top and rerun. Mention this check in the report (limitations).

## 2. Split (70/15/15, stratified, fixed seed)

In [ ]:
p_train, p_tmp, y_train, y_tmp = train_test_split(paths, labels, test_size=0.30, stratify=labels, random_state=SEED)
p_val, p_test, y_val, y_test = train_test_split(p_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED)
print('train', len(p_train), 'val', len(p_val), 'test', len(p_test))
pd.DataFrame({'path': np.concatenate([p_train, p_val, p_test]),
              'label': np.concatenate([y_train, y_val, y_test]),
              'split': ['train'] * len(p_train) + ['val'] * len(p_val) + ['test'] * len(p_test)}).to_csv(f'{OUT}/split.csv', index=False)

## 3. Feature extraction (ResNet50 baseline, CLIP ViT-B/16, CLIP ViT-L/14)

In [ ]:
def feats(enc, plist, perturb=None, bs=32):
    out = []
    for i in range(0, len(plist), bs):
        imgs = [load_image(p) for p in plist[i:i + bs]]
        if perturb: imgs = [perturb(im) for im in imgs]
        out.append(enc.encode(imgs))
    return np.concatenate(out)

BASE, B16, L14 = 'ResNet50 (A1 backbone)', 'CLIP ViT-B/16', 'CLIP ViT-L/14'
MAKERS = {BASE: lambda: ResNetEncoder(),
          B16:  lambda: ClipEncoder('ViT-B-16', 'openai'),
          L14:  lambda: ClipEncoder('ViT-L-14', 'openai')}
enc, X = {}, {}
for name, make in MAKERS.items():
    t0 = time.time()
    enc[name] = make()
    X[name] = {s: feats(enc[name], pl) for s, pl in [('train', p_train), ('val', p_val), ('test', p_test)]}
    print(f'{name}: dim={X[name]["train"].shape[1]}  ({time.time() - t0:.0f}s)')

## 4. Main comparison: baseline vs SOTA
Same classifier head for all backbones (standardise + logistic regression). The regularisation `C` is selected on the **validation** set (log-loss), the test set is used once.

In [ ]:
def fit_probe(Xtr, ytr, Xva, yva, Cs=(1e-4, 1e-3, 1e-2, 1e-1, 1)):
    best = None
    for C in Cs:
        clf = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000)).fit(Xtr, ytr)
        ll = log_loss(yva, clf.predict_proba(Xva), labels=[0, 1])
        if best is None or ll < best[0]: best = (ll, C, clf)
    return best[2], best[1]

def metrics(clf, Xte, yte):
    pr = clf.predict_proba(Xte); pred = pr.argmax(1)
    p, r, f, _ = precision_recall_fscore_support(yte, pred, average='macro', zero_division=0)
    return dict(Accuracy=accuracy_score(yte, pred), Precision=p, Recall=r, F1=f, AUC=roc_auc_score(yte, pr[:, 1]))

clfs, bestC, rows = {}, {}, []
for name in MAKERS:
    clfs[name], bestC[name] = fit_probe(X[name]['train'], y_train, X[name]['val'], y_val)
    rows.append({'Model': name, 'C': bestC[name], **metrics(clfs[name], X[name]['test'], y_test)})
rows.insert(0, {'Model': 'A1 Keras ResNet50 (from A1 report)', 'C': np.nan, 'Accuracy': 0.9375, 'Precision': 0.94, 'Recall': 0.94, 'F1': 0.94, 'AUC': np.nan})
res = pd.DataFrame(rows).set_index('Model').round(3)
res.to_csv(f'{OUT}/main_results.csv')
res

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, name in zip(axes, MAKERS):
    cm = confusion_matrix(y_test, clfs[name].predict(X[name]['test']))
    ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(name)
plt.tight_layout(); savefig('confusion_matrices'); plt.show()

## 5. Ablations
**(a) Backbone:** ResNet50 vs CLIP ViT-B/16 vs CLIP ViT-L/14 (already in the table above).
**(b) Augmentation:** train the probe with and without augmented views of the training images (flip, random crop, small colour jitter). Validation and test are never augmented.

In [ ]:
aug = T.Compose([T.RandomHorizontalFlip(), T.RandomResizedCrop(224, scale=(0.5, 1.0)), T.ColorJitter(0.2, 0.2)])
N_VIEWS = 3
torch.manual_seed(SEED)
abl_rows = []
for name in MAKERS:
    Xa = np.concatenate([X[name]['train']] + [feats(enc[name], p_train, perturb=aug) for _ in range(N_VIEWS)])
    ya = np.tile(y_train, N_VIEWS + 1)
    clf_a, C_a = fit_probe(Xa, ya, X[name]['val'], y_val)
    m0 = metrics(clfs[name], X[name]['test'], y_test); m1 = metrics(clf_a, X[name]['test'], y_test)
    abl_rows.append({'Model': name, 'Acc (no aug)': m0['Accuracy'], 'Acc (aug)': m1['Accuracy'], 'F1 (no aug)': m0['F1'], 'F1 (aug)': m1['F1']})
abl = pd.DataFrame(abl_rows).set_index('Model').round(3)
abl.to_csv(f'{OUT}/ablation_augmentation.csv')
abl

## 6. Learning curves (data efficiency)
A linear probe has no epochs, so instead of loss-vs-epoch curves we plot validation accuracy against the number of training images (3 random subsets per point).
Add your A1 Keras training curves (from the A1 notebook) to the report as the "training curves" for the baseline.

In [ ]:
fracs = [0.1, 0.25, 0.5, 0.75, 1.0]
plt.figure(figsize=(6, 4))
for name in MAKERS:
    means, stds = [], []
    for f in fracs:
        accs = []
        for s in range(3):
            if f < 1.0:
                idx, _ = train_test_split(np.arange(len(y_train)), train_size=f, stratify=y_train, random_state=SEED + s)
            else:
                idx = np.arange(len(y_train))
            clf = make_pipeline(StandardScaler(), LogisticRegression(C=bestC[name], max_iter=5000)).fit(X[name]['train'][idx], y_train[idx])
            accs.append(clf.score(X[name]['val'], y_val))
        means.append(np.mean(accs)); stds.append(np.std(accs))
    m, s_ = np.array(means), np.array(stds)
    plt.plot([int(f * len(y_train)) for f in fracs], m, marker='o', label=name)
    plt.fill_between([int(f * len(y_train)) for f in fracs], m - s_, m + s_, alpha=0.15)
plt.xlabel('Training images'); plt.ylabel('Validation accuracy'); plt.title('Learning curves'); plt.grid(alpha=.3); plt.legend()
savefig('learning_curves'); plt.show()

## 7. Robustness: do the models survive JPEG compression, blur and downscaling?
Real-world images are re-compressed and resized by social networks. Detectors often break here. Models trained on clean images, tested on perturbed test images.

In [ ]:
def jpeg(q):
    def f(im):
        b = io.BytesIO(); im.save(b, 'JPEG', quality=q); b.seek(0); return Image.open(b).convert('RGB')
    return f
def blur(r): return lambda im: im.filter(ImageFilter.GaussianBlur(r))
def down(s): return lambda im: im.resize((max(8, int(im.width * s)), max(8, int(im.height * s))), Image.BILINEAR).resize(im.size, Image.BILINEAR)

CONDS = {'clean': None, 'JPEG q=50': jpeg(50), 'JPEG q=20': jpeg(20), 'Blur r=2': blur(2), 'Downscale x0.5': down(0.5), 'Downscale x0.25': down(0.25)}
rob = {}
for name in MAKERS:
    rob[name] = {c: clfs[name].score(feats(enc[name], p_test, perturb=f) if f else X[name]['test'], y_test) for c, f in CONDS.items()}
rob_df = pd.DataFrame(rob).round(3)
rob_df.to_csv(f'{OUT}/robustness.csv')
rob_df.plot(kind='bar', figsize=(9, 4), ylim=(0.4, 1.02), rot=20); plt.ylabel('Test accuracy'); plt.axhline(0.5, color='gray', ls=':')
plt.title('Robustness to perturbations'); savefig('robustness'); plt.show()
rob_df

## 8. Explainability
**Occlusion heatmap:** a grey square slides over the image; red = regions whose removal lowers the confidence in the predicted class most.
**Nearest neighbours:** the training images closest to the query in CLIP space (shows *why* the probe thinks it is AI or real, and whether it relies on content similarity).

In [ ]:
FINAL = L14
clf_f, enc_f = clfs[FINAL], enc[FINAL]
pred_f = clf_f.predict(X[FINAL]['test'])
wrong = [i for i in range(len(y_test)) if pred_f[i] != y_test[i]]
right = [i for i in range(len(y_test)) if pred_f[i] == y_test[i]]
sel = []
for c in range(2):
    sel += [i for i in right if y_test[i] == c][:1]
sel += wrong[:2]
print('Selected test indices:', sel, '| errors on test:', len(wrong))

fig, axes = plt.subplots(2, len(sel), figsize=(3.2 * len(sel), 6.6), squeeze=False)
for j, i in enumerate(sel):
    img = load_image(p_test[i])
    heat, base = occlusion_heatmap(enc_f, clf_f, img, grid=7, target=int(pred_f[i]))
    ok = 'correct' if pred_f[i] == y_test[i] else 'WRONG'
    axes[0, j].imshow(base); axes[0, j].set_title(f'true: {class_names[y_test[i]]}\npred: {class_names[pred_f[i]]} ({ok})', fontsize=9)
    axes[1, j].imshow(heat_overlay(base, heat)); axes[1, j].set_title('occlusion heatmap', fontsize=9)
    for a in axes[:, j]: a.axis('off')
plt.tight_layout(); savefig('explain_occlusion'); plt.show()

In [ ]:
G = X[FINAL]['train']
fig, axes = plt.subplots(2, 6, figsize=(15, 5.4))
for r, i in enumerate(sel[:2]):
    q = X[FINAL]['test'][i]
    idx, sims = nearest_neighbors(q, G, k=5)
    axes[r, 0].imshow(ImageOps.fit(load_image(p_test[i]), (224, 224))); axes[r, 0].set_title(f'QUERY ({class_names[y_test[i]]})', fontsize=9)
    for k, (j, s) in enumerate(zip(idx, sims)):
        axes[r, k + 1].imshow(ImageOps.fit(load_image(p_train[j]), (224, 224)))
        axes[r, k + 1].set_title(f'{class_names[y_train[j]]}  sim={s:.2f}', fontsize=9)
for a in axes.ravel(): a.axis('off')
plt.tight_layout(); savefig('explain_neighbors'); plt.show()

## 9. Latency and environment

In [ ]:
test_imgs = [load_image(p) for p in p_test]

def bench(encoder, clf, imgs, n=30):
    ts = []
    for im in imgs[:n]:
        t0 = time.perf_counter()
        clf.predict_proba(encoder.encode([im]))
        if torch.cuda.is_available(): torch.cuda.synchronize()
        ts.append((time.perf_counter() - t0) * 1000)
    ts = ts[3:]
    return np.mean(ts), np.std(ts)

lat = []
env = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
if torch.cuda.is_available():
    for name in MAKERS:
        m, s = bench(enc[name], clfs[name], test_imgs); lat.append({'Model': name, 'Device': env, 'Latency ms (mean)': round(m, 1), 'std': round(s, 1)})
    enc[FINAL].move('cpu')
    m, s = bench(enc[FINAL], clfs[FINAL], test_imgs, n=15); lat.append({'Model': FINAL, 'Device': 'Colab CPU', 'Latency ms (mean)': round(m, 1), 'std': round(s, 1)})
    enc[FINAL].move('cuda')
else:
    for name in MAKERS:
        m, s = bench(enc[name], clfs[name], test_imgs, n=15); lat.append({'Model': name, 'Device': 'CPU', 'Latency ms (mean)': round(m, 1), 'std': round(s, 1)})
t0 = time.perf_counter(); occlusion_heatmap(enc[FINAL], clfs[FINAL], test_imgs[0]); t_occ = (time.perf_counter() - t0) * 1000
lat_df = pd.DataFrame(lat); lat_df.to_csv(f'{OUT}/latency.csv', index=False)
print(f'Occlusion explanation (49 forward passes) on {env}: {t_occ:.0f} ms')
lat_df

## 10. (Optional but recommended) Cross-generator test with a new SOTA generator: SDXL-Turbo
The probe never saw images from this generator. We generate 60 synthetic images offline (Colab), label them clearly as synthetic, and measure how many are detected as AI.
All people in the prompts are generic, no real persons. Generated images are saved to a *separate* folder so they do not mix with the A1 dataset.

In [ ]:
RUN_GENERATION = True
GEN_DIR = '/content/drive/MyDrive/ai_vs_real_extra_sdxl_turbo'
if RUN_GENERATION:
    !pip -q install diffusers transformers accelerate
    from diffusers import AutoPipelineForText2Image
    os.makedirs(GEN_DIR, exist_ok=True)
    pipe = AutoPipelineForText2Image.from_pretrained('stabilityai/sdxl-turbo', torch_dtype=torch.float16, variant='fp16').to('cuda')
    subjects = ['a cup of coffee on a wooden table', 'a portrait of an elderly woman smiling', 'a busy street market in the evening',
                'a bowl of fresh fruit on a kitchen counter', 'a golden retriever in a park', 'a modern kitchen interior',
                'a mountain lake at sunrise', 'a teenager reading a book in a library', 'a plate of pasta with basil', 'a city bus on a rainy street']
    styles = ['realistic photo, natural light', 'candid smartphone photo', 'documentary photography']
    k = 0
    for s in subjects:
        for st in styles:
            for rep in range(2):
                g = torch.Generator('cuda').manual_seed(1000 + k)
                im = pipe(f'{s}, {st}', num_inference_steps=2, guidance_scale=0.0, generator=g).images[0]
                im.save(f'{GEN_DIR}/sdxlturbo_{k:03d}.jpg', quality=95); k += 1
    del pipe; torch.cuda.empty_cache()
    print('generated', k, 'images ->', GEN_DIR)

In [ ]:
gen_paths = sorted(os.path.join(GEN_DIR, f) for f in os.listdir(GEN_DIR) if f.endswith('.jpg'))
xg = []
cg_rows = []
for name in MAKERS:
    Xg = feats(enc[name], gen_paths)
    det = (clfs[name].predict(Xg) == AI_IDX).mean()
    cg_rows.append({'Model': name, 'Detected as AI (SDXL-Turbo, unseen generator)': round(float(det), 3), 'n': len(gen_paths)})
cg = pd.DataFrame(cg_rows).set_index('Model'); cg.to_csv(f'{OUT}/cross_generator.csv'); cg

## 11. Export artifacts for the web app

In [ ]:
ART = '/content/artifacts'; os.makedirs(f'{ART}/gallery', exist_ok=True)
joblib.dump(dict(clf=clfs[L14], class_names=class_names, ai_idx=AI_IDX, arch='ViT-L-14', pretrained='openai'), f'{ART}/probe.joblib')
files = []
for i, p in enumerate(p_train):
    fn = f'{i:04d}.jpg'; ImageOps.fit(load_image(p), (224, 224)).save(f'{ART}/gallery/{fn}', quality=90); files.append(fn)
np.savez(f'{ART}/gallery.npz', emb=X[L14]['train'].astype(np.float32), labels=y_train, files=np.array(files))
shutil.make_archive('/content/artifacts', 'zip', ART)
from google.colab import files as colab_files
colab_files.download('/content/artifacts.zip')

Unzip `artifacts.zip` into the repo folder as `artifacts/` (next to `app.py`) and run `python app.py`.
**Privacy:** the gallery contains training photos. If they show other people, get consent or remove the gallery before publishing the repo.